# SEACO-6161 · 02 Cleanup · `dbw-arc-merone-pipeline-prod`

Runs **one step at a time**, chosen with the `step` widget. Every step:

- is a **dry run by default**. Set `dry_run = false` to act.
- refuses to act unless `confirm_workspace` is exactly `dbw-arc-merone-pipeline-prod`.
- refuses to act on an inventory older than 7 days, so run `01_investigation` again first.
- writes every action, dry or real, to `storage_audit_seaco_6161.cleanup_log`.

| Step | What it does | Can it be undone? |
|---|---|---|
| `vacuum_dry_run` | `VACUUM … DRY RUN` on the candidates, and shows the estimate | nothing changes |
| `vacuum` | `VACUUM … RETAIN 168 HOURS` on the candidates (largest overhead first) | **no**, time travel older than 7 days is gone |
| `set_retention_properties` | writes the 7-day / 30-day retention properties on active root tables | yes |
| `drop_tables` | `DROP TABLE` for the names in `drop_tables` only | **no** (hive_metastore has no UNDROP) |
| `delete_orphan_dirs` | deletes the paths in `orphan_dirs`, which must be in the latest orphan list | **no** |
| `prune_cluster_logs` | deletes cluster-log directories with no file newer than N days | **no** |

**Before you run `vacuum` in this workspace:** tell the owners of the busy writer jobs and avoid running at
the same time as them. A VACUUM with 168 hours only removes files that have been unreferenced for 7 days or
more, so it cannot hit what a running job is writing. It can still slow the job down.
### Drop candidates

None known yet. Take them from `01_investigation` section 7 once the owners agree.

In [0]:
%run ./00_config_and_helpers

In [0]:
STEPS = ["vacuum_dry_run", "vacuum", "set_retention_properties", "drop_tables", "delete_orphan_dirs",
         "prune_cluster_logs"]
dbutils.widgets.dropdown("step", "vacuum_dry_run", STEPS, "1. Step")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "2. Dry run")
dbutils.widgets.text("confirm_workspace", "", "3. Type the workspace name to act")
dbutils.widgets.text("retain_hours", "168", "4. VACUUM retain hours (>= 168)")
dbutils.widgets.text("vacuum_min_overhead_gib", "50", "5. VACUUM tables with overhead >= GiB")
dbutils.widgets.text("vacuum_tables", "", "6. VACUUM only these (db.table, comma list)")
dbutils.widgets.text("vacuum_max_tables", "10", "7. VACUUM at most N tables per run")
dbutils.widgets.text("drop_tables", "", "8. DROP these tables (db.table, comma list)")
dbutils.widgets.dropdown("drop_external_data", "false", ["true", "false"], "9. Also delete external table files")
dbutils.widgets.dropdown("allow_recent", "false", ["true", "false"], "10. Allow drop of recently written tables")
dbutils.widgets.text("orphan_dirs", "", "11. Orphan dirs to delete (comma list)")
dbutils.widgets.text("cluster_log_retention_days", "14", "12. Keep cluster logs newer than N days")

STEP = dbutils.widgets.get("step")
DRY_RUN = dbutils.widgets.get("dry_run") != "false"
CONFIRMED = dbutils.widgets.get("confirm_workspace").strip() == WORKSPACE_NAME
RETAIN_HOURS = int(dbutils.widgets.get("retain_hours"))
MIN_OVERHEAD = float(dbutils.widgets.get("vacuum_min_overhead_gib")) * GIB
VACUUM_ONLY = [t.strip() for t in dbutils.widgets.get("vacuum_tables").split(",") if t.strip()]
VACUUM_MAX = int(dbutils.widgets.get("vacuum_max_tables"))
DROP_LIST = [t.strip() for t in dbutils.widgets.get("drop_tables").split(",") if t.strip()]
DROP_EXTERNAL_DATA = dbutils.widgets.get("drop_external_data") == "true"
ALLOW_RECENT = dbutils.widgets.get("allow_recent") == "true"
ORPHAN_LIST = [p.strip() for p in dbutils.widgets.get("orphan_dirs").split(",") if p.strip()]
LOG_DAYS = int(dbutils.widgets.get("cluster_log_retention_days"))

In [0]:
# ---- Safety gates -----------------------------------------------------------------------------------------
assert RETAIN_HOURS >= MIN_RETAIN_HOURS, f"retain_hours must be >= {MIN_RETAIN_HOURS}"
assert spark.conf.get("spark.databricks.delta.retentionDurationCheck.enabled", "true") == "true", \
    "retentionDurationCheck has been disabled on this cluster; re-enable it"
if STEP != "vacuum_dry_run" and not DRY_RUN and not CONFIRMED:
    raise Exception(f"Refusing to act: confirm_workspace must be exactly '{WORKSPACE_NAME}' "
                    f"(this notebook runs on {WORKSPACE_URL})")

INV, INV_TS = latest("table_inventory")
age_days = (now_utc() - INV_TS).total_seconds() / 86400
print(f"Using inventory from {INV_TS:%Y-%m-%d %H:%M} UTC ({age_days:.1f} days old)")
if not DRY_RUN and age_days > MAX_AUDIT_AGE_DAYS:
    raise Exception(f"Inventory is older than {MAX_AUDIT_AGE_DAYS} days; re-run 01_investigation first")

CLONE_SOURCE_NAMES = {r["full_name"] for r in INV.filter("is_shallow_clone_source").select("full_name").collect()}
print(f"Step: {STEP} | dry_run={DRY_RUN} | confirmed={CONFIRMED} | clone sources protected: {len(CLONE_SOURCE_NAMES)}")

## Step: `vacuum_dry_run` / `vacuum`

In [0]:
def vacuum_candidates():
    c = INV.filter((F.col("provider") == "delta") & (F.col("storage_class") == "root") & F.col("error").isNull())
    if VACUUM_ONLY:
        c = c.filter(F.col("full_name").isin(VACUUM_ONLY))
    else:
        c = c.filter(F.col("overhead_bytes") >= MIN_OVERHEAD)
    c = c.filter(~F.col("full_name").isin(DROP_LIST))  # no point vacuuming what is about to be dropped
    return [r.asDict() for r in c.orderBy(F.desc("overhead_bytes")).collect()]


def run_vacuum(dry):
    cands = vacuum_candidates()
    blocked = [c for c in cands if c["full_name"] in CLONE_SOURCE_NAMES]
    cands = [c for c in cands if c["full_name"] not in CLONE_SOURCE_NAMES][:VACUUM_MAX]
    for b in blocked:
        print(f"  SKIP (shallow-clone source of {b['shallow_clones']}): {b['full_name']}")
    print(f"{len(cands)} tables, estimated overhead {human(sum(c['overhead_bytes'] or 0 for c in cands))}\n")
    spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
    freed_total = 0
    for c in cands:
        db, t = c["full_name"].split(".", 1)
        name = qname(db, t)
        t0 = time.time()
        try:
            if dry:
                n = spark.sql(f"VACUUM {name} RETAIN {RETAIN_HOURS} HOURS DRY RUN").count()
                msg = f"dry run lists {n:,} files (Databricks caps the list at 1,000)"
                print(f"  {c['full_name']:<60} est. {human(c['overhead_bytes']):>12}  {msg}")
                log_action("vacuum_dry_run", c["full_name"], True, "ok", c["disk_bytes"], None, time.time() - t0, msg)
            else:
                before = content_summary(c["location"])["bytes"]
                spark.sql(f"VACUUM {name} RETAIN {RETAIN_HOURS} HOURS")
                after = content_summary(c["location"])["bytes"]
                freed = (before or 0) - (after or 0)
                freed_total += freed
                print(f"  {c['full_name']:<60} freed {human(freed):>12} in {time.time() - t0:,.0f}s")
                log_action("vacuum", c["full_name"], False, "ok", before, after, time.time() - t0)
        except Exception as e:
            print(f"  ! {c['full_name']}: {str(e)[:300]}")
            log_action(STEP, c["full_name"], dry, "error", detail=str(e))
    if not dry:
        print(f"\nTotal freed: {human(freed_total)}")


if STEP in ("vacuum_dry_run", "vacuum"):
    run_vacuum(dry=(STEP == "vacuum_dry_run" or DRY_RUN))

## Step: `set_retention_properties`

In [0]:
RETENTION_PROPS = {"delta.deletedFileRetentionDuration": "interval 7 days",
                   "delta.logRetentionDuration": "interval 30 days"}

if STEP == "set_retention_properties":
    active = (INV.filter((F.col("provider") == "delta") & (F.col("storage_class") == "root")
                         & (F.col("last_modified") >= F.lit(now_utc() - dt.timedelta(days=RECENT_DAYS))))
              .select("full_name", "deleted_file_retention", "log_retention").collect())
    props = ", ".join(f"'{k}' = '{v}'" for k, v in RETENTION_PROPS.items())
    for r in active:
        if r["deleted_file_retention"] == RETENTION_PROPS["delta.deletedFileRetentionDuration"] \
                and r["log_retention"] == RETENTION_PROPS["delta.logRetentionDuration"]:
            continue
        db, t = r["full_name"].split(".", 1)
        stmt = f"ALTER TABLE {qname(db, t)} SET TBLPROPERTIES ({props})"
        if DRY_RUN:
            print("DRY:", stmt)
        else:
            try:
                spark.sql(stmt)
                log_action(STEP, r["full_name"], False, "ok", detail=props)
                print("set:", r["full_name"])
            except Exception as e:
                log_action(STEP, r["full_name"], False, "error", detail=str(e))
                print(f"  ! {r['full_name']}: {str(e)[:200]}")

## Step: `drop_tables`

In [0]:
def drop_one(full_name):
    db, t = full_name.split(".", 1)
    try:
        ext = table_extended(db, t)
    except Exception as e:
        print(f"  ! {full_name}: not found ({str(e)[:120]})")
        return
    ttype, loc = ext.get("Type"), ext.get("Location")
    sclass = classify_path(loc)
    detail = {}
    if (ext.get("Provider") or "").lower() == "delta":
        d = describe_detail(db, t)
        detail = dict(last_modified=d.get("lastModified"), live=d.get("sizeInBytes"))
    if full_name in CLONE_SOURCE_NAMES:
        print(f"  REFUSED {full_name}: it is a shallow-clone source; drop or deep-clone its clones first")
        return
    lm = detail.get("last_modified")
    if lm and lm >= now_utc() - dt.timedelta(days=RECENT_DAYS) and not ALLOW_RECENT:
        print(f"  REFUSED {full_name}: written {lm:%Y-%m-%d} (within {RECENT_DAYS} days); set allow_recent=true if intended")
        return
    size = content_summary(loc)["bytes"] if sclass == "root" else None
    plan = f"{ttype} {sclass} {human(size)} at {loc}"
    if DRY_RUN:
        print(f"  DRY drop {full_name}: {plan}")
        log_action("drop_tables", full_name, True, "planned", size, None, None, plan)
        return
    t0 = time.time()
    spark.sql(f"DROP TABLE {qname(db, t)}")
    if ttype == "EXTERNAL" and DROP_EXTERNAL_DATA and sclass == "root" and loc:
        dbutils.fs.rm(loc, True)
    left = content_summary(loc)["bytes"] if sclass == "root" and loc and path_exists(loc) else 0
    print(f"  dropped {full_name}: {plan}; bytes left at location: {human(left)}")
    log_action("drop_tables", full_name, False, "ok", size, left, time.time() - t0, plan)


if STEP == "drop_tables":
    if not DROP_LIST:
        print("drop_tables widget is empty; nothing to do")
    for name in DROP_LIST:
        drop_one(name)

## Step: `delete_orphan_dirs`

In [0]:
if STEP == "delete_orphan_dirs":
    known, _ = latest("orphan_dirs")
    known = {normalize(r["path"]): r for r in known.collect()}
    live_locs = [normalize(r["location"]) for r in INV.select("location").collect() if r["location"]]
    for p in ORPHAN_LIST:
        n = normalize(p)
        if n not in known:
            print(f"  REFUSED {p}: not in the latest orphan_dirs audit")
            continue
        if classify_path(p) != "root" or any(is_under(l, n) or is_under(n, l) for l in live_locs):
            print(f"  REFUSED {p}: not on root storage, or overlaps a registered table location")
            continue
        size = content_summary(p)["bytes"]
        if DRY_RUN:
            print(f"  DRY rm -r {p}  ({human(size)})")
            log_action(STEP, p, True, "planned", size)
        else:
            t0 = time.time()
            dbutils.fs.rm(p, True)
            print(f"  removed {p}  ({human(size)})")
            log_action(STEP, p, False, "ok", size, 0, time.time() - t0)

## Step: `prune_cluster_logs`

In [0]:
if STEP == "prune_cluster_logs":
    confs, _ = latest("cluster_log_conf")
    dests = sorted({r["destination"] for r in confs.filter("storage_class = 'root'").collect() if r["destination"]})
    cutoff = now_utc() - dt.timedelta(days=LOG_DAYS)
    print(f"{len(dests)} root log destinations; removing cluster dirs with no file newer than {cutoff:%Y-%m-%d}")
    freed = 0
    for dest in dests:
        for cdir in ls(dest):
            if not cdir.isDir():
                continue
            newest = newest_mtime(cdir.path)
            if newest and newest >= cutoff:
                continue
            size = content_summary(cdir.path)["bytes"] or 0
            freed += size
            if DRY_RUN:
                print(f"  DRY rm -r {cdir.path}  ({human(size)}, newest {newest})")
            else:
                dbutils.fs.rm(cdir.path, True)
            log_action(STEP, cdir.path, DRY_RUN, "planned" if DRY_RUN else "ok", size, 0)
    print(f"{'Would free' if DRY_RUN else 'Freed'}: {human(freed)}")

## What has been done so far

In [0]:
%sql
SELECT step, dry_run, status, count(*) AS actions,
       round(sum(bytes_before - coalesce(bytes_after, bytes_before)) / power(1024,4), 3) AS freed_tib,
       min(action_ts) AS first, max(action_ts) AS last
FROM storage_audit_seaco_6161.cleanup_log
GROUP BY step, dry_run, status
ORDER BY last DESC